In [ ]:
import shap
from scipy.stats import spearmanr
from itertools import combinations
import os
import pickle
import numpy as np
import inspect
import shap


In [ ]:
def get_shap_dict(dir, type=None):
    if type is not None:
        shap_results_fold_dir = os.path.join(dir, f'shap_all_test_{type}.pkl')
    elif type is None:
        shap_results_fold_dir = os.path.join(dir, f'shap_all_test.pkl')
    shap_dict = pickle.load(open(shap_results_fold_dir, 'rb'))     
    shap_values = np.sum(shap_dict['shap values'], axis=2)
    feature_names = list(shap_dict['Feature names'])
    return shap_values, feature_names

In [ ]:
# For each fold
type='kirc'
dict = {
    'brca': ['64', '128', '192', '256', '320', '384', '448', 'seed_1', '576', '640', '704'],
    'blca': ['64', '128', '192', 'seed_1'],
    'luad': ['64', '128', '192', '256', 'seed_1'],
    'kirc': ['64', '128', 'seed_1']
}
dir = f'../results/ablations/dss_survival_{type}/DIMAFx/'
for repr_type in ['modal', 'post_attn', 'post_attn_av']:
    print(f"Representation type: {repr_type}")
    for i in range(5):
        print("Fold: ", i)
        for combo in combinations(dict[type], 2):
            print(f"Comparing {combo[0]} vs {combo[1]}")
            results = {}
            fold_dir = os.path.join(dir, f'Fold_{i}/post_training/shap/{repr_type}')
            shap_values_1, _ = get_shap_dict(fold_dir, type=f"{combo[0]}")
            shap_values_2, _ = get_shap_dict(fold_dir, type=f"{combo[1]}")

            importance_1 = np.mean(np.abs(shap_values_1), axis=0)
            importance_2 = np.mean(np.abs(shap_values_2), axis=0)

            # Spearman rank correlation
            r, p = spearmanr(importance_1, importance_2)
            print(f"Spearman rank correlation: {r:.3f}, p={p:.3e}")
            if p >= 0.05:
                print(f"Correlation {r} is not significant (p={p:.3e})")
            if r < min1:
                min1 = r

            # [n_test_samples, n_feats] after summing over embed dim
            importance_1_per_sample = np.abs(shap_values_1)
            importance_2_per_sample = np.abs(shap_values_2)

            # Correlation per sample
            correlations = []
            for s in range(importance_1_per_sample.shape[0]):
                r, p = spearmanr(importance_1_per_sample[s], importance_2_per_sample[s])
                correlations.append(r)
                if p >= 0.05:
                    print(f"Correlation for sample {s} is not significant (p={p:.3e})")
                    print()
                if r < min3:
                    min3 = r
                

            correlations = np.array(correlations)
            print(f"Mean Spearman per sample: {np.mean(np.abs(correlations)):.3f} ± {np.std(np.abs(correlations)):.3f}")
            print(f"Min: {np.min(correlations):.3f}, Max: {np.max(correlations):.3f}")
            if np.mean(np.abs(correlations)) < min2:
                min2 = np.mean(np.abs(correlations))

print(f"Minimum Spearman rank correlation across all combos and folds: {min1:.3f}")
print(f"Minimum Spearman rank correlation per sample across all combos and folds: {min2:.3f}")
print(f"Minimum Spearman rank correlation per sample across all combos and folds: {min3:.3f}")



In [ ]:
# For each fold
type='brca'
dir = f'../results/ablations/dss_survival_{type}/DIMAFx/'
min1 = 1
min2 = 1
min3 = 1
for repr_type in ['modal', 'post_attn']:
    for i in range(5):
        for combo in combinations([1, 2, 3, 4, 5], 2):
            print("Combo: ", combo)
            
            print("Fold: ", i)
            results = {}
            fold_dir = os.path.join(dir, f'Fold_{i}/post_training/shap/{repr_type}')
            shap_values_1, _ = get_shap_dict(fold_dir, type=f"seed_{combo[0]}")
            shap_values_2, _ = get_shap_dict(fold_dir, type=f"seed_{combo[1]}")

            importance_1 = np.mean(np.abs(shap_values_1), axis=0)
            importance_2 = np.mean(np.abs(shap_values_2), axis=0)

            # Spearman rank correlation
            r, p = spearmanr(importance_1, importance_2)
            print(f"Spearman rank correlation: {r:.3f}, p={p:.3e}")
            if p >= 0.05:
                print(f"Correlation {r} is not significant (p={p:.3e})")
            if r < min1:
                min1 = r

            # [n_test_samples, n_feats] after summing over embed dim
            importance_1_per_sample = np.abs(shap_values_1)
            importance_2_per_sample = np.abs(shap_values_2)

            # Correlation per sample
            correlations = []
            for s in range(importance_1_per_sample.shape[0]):
                r, p = spearmanr(importance_1_per_sample[s], importance_2_per_sample[s])
                correlations.append(r)
                if p >= 0.05:
                    print(f"Correlation for sample {s} is not significant (p={p:.3e})")
                    print()
                if r < min3:
                    min3 = r
                

            correlations = np.array(correlations)
            print(f"Mean Spearman per sample: {np.mean(np.abs(correlations)):.3f} ± {np.std(np.abs(correlations)):.3f}")
            print(f"Min: {np.min(correlations):.3f}, Max: {np.max(correlations):.3f}")
            if np.mean(np.abs(correlations)) < min2:
                min2 = np.mean(np.abs(correlations))

print(f"Minimum Spearman rank correlation across all combos and folds: {min1:.3f}")
print(f"Minimum Spearman rank correlation per sample across all combos and folds: {min2:.3f}")
print(f"Minimum Spearman rank correlation per sample across all combos and folds: {min3:.3f}")
